# LLM Evaluation on Google Colab

Notebook này dùng riêng cho Google Colab để benchmark các LLM trên comparative quintuple extraction.

Trước khi chạy:
1. Thêm `OPENROUTER_API_KEY` vào Colab Secrets.
2. Clone project từ GitHub hoặc upload `project.zip`.
3. Chỉnh `DATASETS`, `SPLIT`, `MODELS`, `PROMPT_STRATEGY` ở các cell cấu hình.

## Nên dùng openrouter hay hf-local?

- Dùng `openrouter` khi:
  - Bạn muốn chạy nhanh, ổn định, không phụ thuộc VRAM local.
  - Bạn cần benchmark nhiều model thương mại trong cùng một pipeline.
  - Bạn chấp nhận chi phí API theo token.
- Dùng `hf-local` khi:
  - Bạn muốn tiết kiệm chi phí API (đổi lại tốn tài nguyên GPU runtime).
  - Bạn chỉ cần test các model open-source trên Hugging Face.
  - Bạn muốn toàn quyền kiểm soát model/quantization.

## Gợi ý VRAM cho hf-local (tham khảo)

- 3B model:
  - FP16/BF16: >= 8 GB VRAM
  - 4-bit: >= 4-6 GB VRAM
- 7B-8B model:
  - FP16/BF16: >= 16 GB VRAM
  - 4-bit: >= 8-12 GB VRAM
- 13B model:
  - FP16/BF16: >= 24 GB VRAM
  - 4-bit: >= 12-16 GB VRAM
- 30B+ model:
  - Thường cần multi-GPU hoặc quantization mạnh, không phù hợp Colab free.

Lưu ý:
- Colab free thường phù hợp nhất với `hf-local` + 4-bit + model <= 7B/8B.
- Nếu OOM, giảm kích thước model trước khi giảm `MAX_OUTPUT_TOKENS`.
- Với benchmark lớn, nên chạy `--limit` nhỏ để kiểm tra throughput và bộ nhớ trước.

## Model gợi ý từ Hugging Face (EN + VI)

Shortlist cân bằng chất lượng/chi phí. Đây là instruction-tuned models, bám format tốt hơn base model, có hiệu năng ổn trên tác bụ có ràng buộc output, khả năng đa ngôn ngữ tốt
1. `Qwen/Qwen2.5-3B-Instruct` (tiết kiệm VRAM)
2. `Qwen/Qwen2.5-7B-Instruct`
3. `Qwen/Qwen2.5-14B-Instruct`
4. `mistralai/Mistral-7B-Instruct-v0.3`
5. `meta-llama/Llama-3.1-8B-Instruct`
6. `google/gemma-2-9b-it`
7. `microsoft/Phi-3-medium-4k-instruct` (tiết kiệm VRAM)


In [ ]:
# Cell 1 · Colab paths
import os
import sys

assert 'google.colab' in sys.modules or os.path.exists('/content'), 'Notebook này chỉ dành cho Google Colab.'

WORK_DIR = '/content/msc-project'
LLMEVAL_DIR = os.path.join(WORK_DIR, 'llm_eval')
DATASETS_ROOT = os.path.join(WORK_DIR, 'datasets')
OUTPUT_DIR = os.path.join(LLMEVAL_DIR, 'results')
CACHE_DIR = os.path.join(LLMEVAL_DIR, 'cache')

print('Project root :', WORK_DIR)
print('LLM eval dir :', LLMEVAL_DIR)

In [ ]:
# Cell 2 · Clone repo hoặc upload project.zip
GITHUB_REPO = ''  # ví dụ: 'https://github.com/haiyan/msc-project.git'

import os
import shutil
import subprocess
import zipfile

if not os.path.exists(LLMEVAL_DIR):
    if GITHUB_REPO:
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, WORK_DIR], check=True)
        print('Cloned from GitHub.')
    else:
        from google.colab import files
        print('Upload project.zip below:')
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
        with zipfile.ZipFile(zip_name) as zf:
            zf.extractall('/content/')
        extracted = [
            d for d in os.listdir('/content/')
            if os.path.isdir(f'/content/{d}') and d != 'sample_data' and d != 'msc-project'
        ]
        if extracted and not os.path.exists(WORK_DIR):
            shutil.move(f'/content/{extracted[0]}', WORK_DIR)
        print(f'Extracted to {WORK_DIR}')
else:
    print(f'Project already exists at {WORK_DIR}')

In [ ]:
# Cell 3 · Install dependencies
import os
import subprocess
import sys

reqs = os.path.join(LLMEVAL_DIR, 'requirements.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', reqs, '-q'], check=True)
print('Dependencies installed.')

In [ ]:
# Cell 4 · Load OpenRouter API key from Colab Secrets
import os
from google.colab import userdata

os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY not found in Colab Secrets.'
print('API key loaded.')

## Khối A: HF-Local Workflow (Sử dụng mô hình cục bộ - tiết kiệm chi phí)
Chạy các ô dưới đây nếu bạn muốn sử dụng HuggingFace models cục bộ.

In [ ]:
# [A.1] HF-Local Configuration
print("=" * 100)
print("KHỐI A: HF-LOCAL WORKFLOW")
print("=" * 100)

DATASETS = 'camera-coqe,vcom-data'
SPLIT = 'test'
PROMPT_STRATEGY = 'few-shot'  # zero-shot | few-shot | cot

# Provider settings for HF-Local
PROVIDER = 'hf-local'
HF_DTYPE = 'auto'             # auto | float16 | bfloat16
HF_LOAD_IN_4BIT = False       # Set to True if VRAM is limited

# HF-Local Model recommendations:
# - Qwen/Qwen2.5-3B-Instruct (3B, lightweight, ~7-8GB)
# - Qwen/Qwen2.5-7B-Instruct (7B, balanced, ~15-16GB)
# - Qwen/Qwen2.5-14B-Instruct (14B, powerful, ~28-30GB)
# - mistralai/Mistral-7B-Instruct-v0.2 (7B, fast, ~15-16GB)

MODELS = [
    'Qwen/Qwen2.5-3B-Instruct',
    'Qwen/Qwen2.5-7B-Instruct',
]

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 256
SLEEP_SECONDS = 0.3
LIMIT = 0

print('Configuration ready for HF-Local.')
print(f'Provider: {PROVIDER}')
print(f'Models: {MODELS}')
print(f'Strategy: {PROMPT_STRATEGY}')
print(f'HF_DTYPE: {HF_DTYPE}, HF_LOAD_IN_4BIT: {HF_LOAD_IN_4BIT}')

In [ ]:
# [A.2] HF-Local: Single-sentence smoke test
import os
import sys

SMOKE_SENTENCE = 'Bên cạnh đó, iPhone 14 được nâng cấp bộ nhớ lên đến 6GB RAM cao hơn iPhone 13 đến 2GB RAM, cho khả năng đa nhiệm tốt hơn.'
SMOKE_DATASET = 'vcom-data'      # vcom-data | camera-coqe
SMOKE_LANGUAGE = 'vi'            # vi | en | auto
SMOKE_STRATEGY = PROMPT_STRATEGY

sys.path.insert(0, LLMEVAL_DIR)
from prompts import build_messages
from client import HuggingFaceLocalClient

messages = build_messages(
    sentence=SMOKE_SENTENCE,
    language=SMOKE_LANGUAGE,
    dataset=SMOKE_DATASET,
    strategy=SMOKE_STRATEGY,
)

print(f"\n[HF-Local Smoke Test] Testing with model: {MODELS[0]}")
smoke_client = HuggingFaceLocalClient(
    model=MODELS[0],
    temperature=TEMPERATURE,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    dtype=HF_DTYPE,
    load_in_4bit=HF_LOAD_IN_4BIT,
)

smoke_output = smoke_client.generate(messages)

print('Provider :', PROVIDER)
print('Model    :', MODELS[0])
print('Strategy :', SMOKE_STRATEGY)
print('Sentence :', SMOKE_SENTENCE)
print('-' * 100)
print('Output:')
print(smoke_output)

In [ ]:
# [A.3] HF-Local: Full evaluation
import os
import subprocess
import sys

print("\n[HF-Local] Running full evaluation...")
cmd = [
    sys.executable, os.path.join(LLMEVAL_DIR, 'run_eval.py'),
    '--datasets', DATASETS,
    '--split', SPLIT,
    '--models', *MODELS,
    '--provider', PROVIDER,
    '--prompt-strategy', PROMPT_STRATEGY,
    '--temperature', str(TEMPERATURE),
    '--max-output-tokens', str(MAX_OUTPUT_TOKENS),
    '--sleep-seconds', str(SLEEP_SECONDS),
    '--datasets-root', DATASETS_ROOT,
    '--output-dir', OUTPUT_DIR,
    '--cache-dir', CACHE_DIR,
    '--hf-dtype', HF_DTYPE,
]

if HF_LOAD_IN_4BIT:
    cmd += ['--hf-load-in-4bit']

if LIMIT > 0:
    cmd += ['--limit', str(LIMIT)]

print('Running command:')
print(' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=False)
print('Exit code:', result.returncode)

## Khối B: OpenRouter Workflow (Sử dụng API - các mô hình lớn)
Chạy các ô dưới đây nếu bạn muốn sử dụng OpenRouter API để gọi các mô hình mạnh mẽ (GPT-4, Claude, Gemini, v.v.).

In [ ]:
# [B.1] OpenRouter Configuration
print("=" * 100)
print("KHỐI B: OPENROUTER WORKFLOW")
print("=" * 100)

DATASETS = 'camera-coqe,vcom-data'
SPLIT = 'test'
PROMPT_STRATEGY = 'few-shot'  # zero-shot | few-shot | cot

# Provider settings for OpenRouter
PROVIDER = 'openrouter'
# OpenRouter API key should already be loaded in Colab Secrets (OPENROUTER_API_KEY)

# OpenRouter model options (via router or direct):
# - openai/gpt-4o-mini (fast, cost-effective)
# - anthropic/claude-3.5-haiku (multimodal, reliable)
# - google/gemini-2.0-flash-001 (fast, powerful)
# - deepseek/deepseek-chat (efficient)
# - qwen/qwen-2.5-72b-instruct (very powerful)
# - meta-llama/llama-3.3-70b-instruct (open-source alternative)

MODELS = [
    'openai/gpt-4o-mini',
    'anthropic/claude-3.5-haiku',
    'google/gemini-2.0-flash-001',
]

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 256
SLEEP_SECONDS = 0.3  # Respect rate limits
LIMIT = 0

print('Configuration ready for OpenRouter.')
print(f'Provider: {PROVIDER}')
print(f'Models: {MODELS}')
print(f'Strategy: {PROMPT_STRATEGY}')
print('WARNING: OpenRouter requires OPENROUTER_API_KEY secret to be set!')

In [ ]:
# [B.2] OpenRouter: Single-sentence smoke test
import os
import sys

SMOKE_SENTENCE = 'Bên cạnh đó, iPhone 14 được nâng cấp bộ nhớ lên đến 6GB RAM cao hơn iPhone 13 đến 2GB RAM, cho khả năng đa nhiệm tốt hơn.'
SMOKE_DATASET = 'vcom-data'      # vcom-data | camera-coqe
SMOKE_LANGUAGE = 'vi'            # vi | en | auto
SMOKE_STRATEGY = PROMPT_STRATEGY

sys.path.insert(0, LLMEVAL_DIR)
from prompts import build_messages
from client import OpenAICompatibleClient

messages = build_messages(
    sentence=SMOKE_SENTENCE,
    language=SMOKE_LANGUAGE,
    dataset=SMOKE_DATASET,
    strategy=SMOKE_STRATEGY,
)

print(f"\n[OpenRouter Smoke Test] Testing with model: {MODELS[0]}")
smoke_client = OpenAICompatibleClient(
    model=MODELS[0],
    base_url='https://openrouter.ai/api/v1',
    api_key_env='OPENROUTER_API_KEY',
    temperature=TEMPERATURE,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

smoke_output = smoke_client.generate(messages)

print('Provider :', PROVIDER)
print('Model    :', MODELS[0])
print('Strategy :', SMOKE_STRATEGY)
print('Sentence :', SMOKE_SENTENCE)
print('-' * 100)
print('Output:')
print(smoke_output)

In [ ]:
# [B.3] OpenRouter: Full evaluation
import os
import subprocess
import sys

print("\n[OpenRouter] Running full evaluation...")
cmd = [
    sys.executable, os.path.join(LLMEVAL_DIR, 'run_eval.py'),
    '--datasets', DATASETS,
    '--split', SPLIT,
    '--models', *MODELS,
    '--provider', PROVIDER,
    '--prompt-strategy', PROMPT_STRATEGY,
    '--temperature', str(TEMPERATURE),
    '--max-output-tokens', str(MAX_OUTPUT_TOKENS),
    '--sleep-seconds', str(SLEEP_SECONDS),
    '--datasets-root', DATASETS_ROOT,
    '--output-dir', OUTPUT_DIR,
    '--cache-dir', CACHE_DIR,
    '--base-url', 'https://openrouter.ai/api/v1',
    '--api-key-env', 'OPENROUTER_API_KEY',
]

if LIMIT > 0:
    cmd += ['--limit', str(LIMIT)]

print('Running command:')
print(' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=False)
print('Exit code:', result.returncode)

## Summary & Download Results
Xem kết quả đánh giá và tải xuống các file kết quả.

In [ ]:
# Cell 7 · Show summary
import json
import pathlib

summary_file = pathlib.Path(OUTPUT_DIR) / f'summary__{SPLIT}.json'

if summary_file.exists():
    with open(summary_file, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    try:
        import pandas as pd
        df = pd.DataFrame([
            {
                'dataset': r['dataset'],
                'model': r['model'],
                'E-T5-MACRO-F1': round(r.get('E-T5-MACRO-F1', 0), 4),
                'E-T4-F1': round(r.get('E-T4-F1', 0), 4),
                'E-CEE-MICRO-F1': round(r.get('E-CEE-MICRO-F1', 0), 4),
            }
            for r in rows
        ]).sort_values(['dataset', 'E-T5-MACRO-F1'], ascending=[True, False])
        print(df.to_string(index=False))
    except ImportError:
        print(rows)
else:
    print('Summary file not found.')

In [ ]:
# Cell 8 · Download results zip
import pathlib
import zipfile
from google.colab import files

results_path = pathlib.Path(OUTPUT_DIR)
zip_path = '/content/llm_eval_results.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in results_path.rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(results_path.parent))

files.download(zip_path)
print('Download started.')